In [54]:
import pandas as pd
import pyarrow.parquet as pq
from collections import defaultdict,Counter

In [55]:
def get_data(path,n=1_00_000):
  data=[]
  parquet=pq.ParquetFile(path)
  for batch in parquet.iter_batches(batch_size=1000,columns=['sentence']):
    if len(data)>=n:
      return data
    df_batch=batch.to_pandas()
    for sentence in df_batch['sentence']:
      if sentence.endswith('.'):
        sentence=sentence[:-1]
      data.append(sentence)
  return data

In [56]:
path="/home/deepakchalla/Desktop/Desktop/NLP/Lab1/sentences.parquet"

In [57]:
data=get_data(path,n=10000)

In [58]:
def punct(c):
  punctuations=[';','!','?',".",",",'"',"'"]
  return c  in punctuations

In [ ]:
from collections import defaultdict, Counter

def byte_pair_encoding(sentences, punct, split_by_vocab=False, vocab_size=32000, merge_steps=32000):
    vocab = defaultdict(int)
    bigrams = defaultdict(int)
    working_sentences = []

    for sentence in sentences:
        for word in sentence.split():
            chars = [c for c in word if not punct(c)]
            if not chars:
                continue
            working_sentences.append(chars)
            for ch in chars:
                vocab[ch] += 1
            for i in range(len(chars) - 1):
                bigrams[(chars[i], chars[i + 1])] += 1

    for _ in range(merge_steps):
        if not bigrams:
            break
        if split_by_vocab and len(vocab) >= vocab_size:
            break

        best = max(bigrams, key=bigrams.get)
        freq = bigrams[best]
        if freq == 0:
            break

        a, b = best
        merged = a + b

        vocab[a] -= freq
        vocab[b] -= freq
        if vocab[a] <= 0:
            del vocab[a]
        if vocab[b] <= 0:
            del vocab[b]
        vocab[merged] += freq

        new_bigrams = Counter()
        for i, chars in enumerate(working_sentences):
            j = 0
            while j < len(chars) - 1:
                if chars[j] == a and chars[j + 1] == b:
                    chars[j:j + 2] = [merged]
                    if j > 0:
                        new_bigrams[(chars[j - 1], chars[j])] += 1
                    if j < len(chars) - 1:
                        new_bigrams[(chars[j], chars[j + 1])] += 1
                else:
                    new_bigrams[(chars[j], chars[j + 1])] += 1
                    j += 1
            working_sentences[i] = chars

        bigrams = new_bigrams

    return list(vocab.keys())


In [61]:
print(byte_pair_encoding(sentences=data,punct=punct,split_by_vocab=True,vocab_size=1000))

['అ', 'మ', 'ె', 'ర', 'క', 'ధ', 'య', 'ష', 'ు', 'డ', 'ొ', 'ల', 'ం', 'ప', 'త', 'భ', 'వ', 'ద', 'ఘ', 'స', 'గ', 'చ', 'ఆ', 'థ', 'ో', 'ీ', 'హ', 'ై', 'ౌ', 'ఇ', 'ణ', 'ే', 'ఎ', 'ఫ', 'ఏ', 'జ', 'శ', 'ఈ', 'బ', 'ఉ', 'ూ', 'ృ', 'ౖ', 'ఖ', 'ళ', 'ఒ', '2', '0', '1', '3', '6', '7', '5', '9', 'ఛ', 'ఊ', 'K', 'B', 'ఠ', 'ఓ', 'ఐ', '4', '8', 'ఢ', 'ఝ', 'C', 'z', 'e', 'c', 'h', 'M', 'W', 'ఋ', 'S', 'u', 'd', 'a', 'n', 's', 'U', 'g', 'ః', '౦', 'k', 'b', 'l', 'w', 't', 'Y', 'R', 'o', 'r', 'P', 'y', 'i', 'F', 'O', 'G', 'V', 'N', 'T', 'E', 'm', 'H', 'ఞ', 'I', 'v', 'p', 'D', 'J', 'L', 'X', 'ఔ', 'A', 'ఙ', '-', 'ఁ', '/', 'f', 'j', 'Z', 'x', '౯', '@', 'Q', 'q', ':', 'ని', 'ార', '్ర', 'న్న', 'స్', 'ర్', 'ంది', 'లో', 'లు', 'కు', '్య', 'ల్', 'ాల', 'ంచ', 'క్', 'గా', 'ట్', 'ను', 'ారు', 'ప్ర', 'రి', 'వి', 'తు', 'చే', 'కి', 'డి', 'డు', 'సి', 'టి', 'మా', 'రా', 'ంత', 'ప్ప', 'త్', 'యి', 'లి', 'రు', 'తి', 'టు', 'ంట', 'కా', 'నా', 'సు', 'వా', 'నే', 'ార్', 'ంచి', 'ంలో', 'లా', 'ద్', 'ండ', 'తో', 'గు', 'ంగా', 'చి', 'కో', 'ది', 'పా', 'చ్', '

In [ ]:
import math
def wordpiece_training(sentences, punct, vocab_size=32000, max_steps=30000):
    vocab = Counter()
    tokenized = []

    for sentence in sentences:
        for word in sentence.split():
            chars = [c for c in word if not punct(c)]
            if not chars:
                continue
            wp = [chars[0]] + ["##" + c for c in chars[1:]]
            tokenized.append(wp)
            for tok in wp:
                vocab[tok] += 1

    vocab = dict(vocab)

    def log_likelihood(vf):
        total = sum(vf.values())
        s = 0.0
        for f in vf.values():
            if f > 0:
                s += f * math.log(f / total)
        return s

    current_score = log_likelihood(vocab)

    def pair_stats(tokenized):
        pf = Counter()
        for wp in tokenized:
            for i in range(len(wp) - 1):
                pf[(wp[i], wp[i+1])] += 1
        return pf

    for _ in range(max_steps):
        if len(vocab) >= vocab_size:
            break

        pairs = pair_stats(tokenized)
        if not pairs:
            break

        best = max(pairs, key=pairs.get)
        a, b = best
        merged = a + b.replace("##", "")

        new_vocab = dict(vocab)
        freq = pairs[best]

        new_vocab[a] -= freq
        new_vocab[b] -= freq
        if new_vocab[a] <= 0: del new_vocab[a]
        if new_vocab[b] <= 0: del new_vocab[b]
        new_vocab[merged] = new_vocab.get(merged, 0) + freq

        new_tokenized = []
        for wp in tokenized:
            nw = []
            i = 0
            while i < len(wp):
                if i < len(wp)-1 and wp[i] == a and wp[i+1] == b:
                    nw.append(merged)
                    i += 2
                else:
                    nw.append(wp[i])
                    i += 1
            new_tokenized.append(nw)

        new_score = log_likelihood(new_vocab)
        if new_score <= current_score:
            break

        vocab = new_vocab
        tokenized = new_tokenized
        current_score = new_score

    return list(vocab.keys())
